# Sequence-generalist Colab Runner

**목표**: 임의 시퀀스 (예: `GROUND→B→A→Y→C`)를 테스트로 줘도 잘 만드는 일반화 모델.

**전략**: env에 추가된 `--random-shape-pool` 옵션으로 **에피소드마다 시퀀스를 랜덤 샘플링**. 정책이 모든 글자와 글자-글자 전환을 다 보게 됨. obs가 shape-agnostic이라 학습된 정책은 어떤 시퀀스에도 적용됨.

**Curriculum** (3 stage):
- Stage 1: pool A~D, 길이 2-3 (워밍업)
- Stage 2: pool A~J, 길이 3-5 (다양화)
- Stage 3: pool A~Z 전체, 길이 4-6 (full coverage)

각 stage가 다음 stage의 `--load-ckpt`로 전이됨.

**Drive 영속**: 모든 ckpt / TB / GIF / eval txt를 처음부터 Google Drive에 직접 저장. 런타임 끊겨도 살아남음.

런타임: `런타임 → 런타임 유형 변경 → GPU (T4)`.

## 1. GitHub clone

In [ ]:
BRANCH = "Saehoon"

!rm -rf RL-2026s1-tp
!git clone https://github.com/umbrellalily/RL-2026s1-tp.git
%cd RL-2026s1-tp

!git fetch origin
!git switch $BRANCH
!git pull origin $BRANCH

!git log --oneline -1
!grep -n 'random_shape_pool' comm_env.py | head -5 || echo '⚠️ random_shape_pool 코드 없음 — fix가 안 됨'

## 2. Google Drive 마운트 + 저장 경로

모든 산출물 Drive 직저장. 런타임 끊겨도 다음 세션에서 mount만 다시 하면 그대로 살아있음.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/drone_results/sequence"
!mkdir -p {DRIVE_ROOT}/ckpts {DRIVE_ROOT}/runs {DRIVE_ROOT}/gifs {DRIVE_ROOT}/evals
print("DRIVE_ROOT =", DRIVE_ROOT)
!ls -la {DRIVE_ROOT}

## 3. 패키지 설치

In [ ]:
!pip install -q torchrl pettingzoo==1.24.3 gymnasium scipy matplotlib pillow tensorboard

## 4. Smoke test: random shape sampling 검증

env가 reset마다 다른 시퀀스를 샘플하는지 확인.

In [ ]:
%cd /content/RL-2026s1-tp

from comm_env import BatteryShapeFormationEnv

env = BatteryShapeFormationEnv(
    grid_size=25, n_agents=14, max_steps=300,
    random_shape_pool=list("ABCDE"),
    random_path_length="2-4",
)

print("5번 reset하면 매번 다른 시퀀스가 나와야 함:")
for i in range(5):
    env.reset(seed=100 + i)
    print(f"  reset #{i}: {env.formation_path.label}")

print("\nobs_dim 일관성 (시퀀스 길이와 무관해야 함):")
for i in range(3):
    obs, _ = env.reset(seed=200 + i)
    print(f"  reset #{i}: obs shape = {obs['drone_0'].shape}, path = {env.formation_path.label}")

## 5. Stage 1 — 워밍업: 작은 pool, 짧은 시퀀스

**Pool**: A, B, C, D (4글자)  
**길이**: 2~3개 타겟 (즉 GROUND→X→Y 또는 GROUND→X→Y→Z)  
**목적**: 기본적인 "좌표 따라가기 + 충돌 회피 + 시퀀스 전환" 정책 형성

총 frames 524K, batch 8192 → 64 iter. ckpt-every 2 → 32개 ckpt.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/sequence"

!python comm_train_battery.py \
  --grid-size 25 --n-agents 14 --max-steps 250 \
  --random-shape-pool "A,B,C,D" --random-path-length "2-3" \
  --shapes GROUND,A \
  --completion-reward 50.0 \
  --assigned-target-reward 0.1 --coverage-delta-reward 0.3 \
  --hover-penalty 0.05 --shaping-coef 0.5 \
  --initial-battery 1.0 --hover-battery-cost 0.002 --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.15 \
  --total-frames 524288 \
  --frames-per-batch 8192 --minibatch-size 512 --ppo-epochs 6 \
  --lr 2e-4 --ent-coef 0.01 --clip-eps 0.15 \
  --ckpt-every 2 \
  --save-dir {DRIVE_ROOT}/ckpts/stage1 \
  --tb-logdir {DRIVE_ROOT}/runs/stage1

## 6. Stage 2 — 다양화: 글자 수와 길이 둘 다 늘림

**Pool**: A~J (10글자)  
**길이**: 3~5개 타겟  
**`--load-ckpt`**: Stage 1의 가장 좋은 ckpt (BEST_ITER_1을 학습 로그 보고 수정)

Stage 2에서는 LR을 1/2로 (1e-4), entropy도 1/2로 (0.005) 낮춰서 사전학습 정책을 부드럽게 확장.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/sequence"
BEST_ITER_1 = 60   # ← Stage 1 학습 로그에서 가장 좋은 iter로 수정

import os
ckpt1 = f"{DRIVE_ROOT}/ckpts/stage1/ckpt_{BEST_ITER_1}.pt"
assert os.path.exists(ckpt1), f"Stage 1 ckpt 없음: {ckpt1}"
print("Stage 1 ckpt 로드:", ckpt1)

!python comm_train_battery.py \
  --grid-size 25 --n-agents 14 --max-steps 400 \
  --random-shape-pool "A,B,C,D,E,F,G,H,I,J" --random-path-length "3-5" \
  --shapes GROUND,A \
  --completion-reward 50.0 \
  --assigned-target-reward 0.1 --coverage-delta-reward 0.3 \
  --hover-penalty 0.05 --shaping-coef 0.5 \
  --initial-battery 1.0 --hover-battery-cost 0.002 --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.15 \
  --load-ckpt {ckpt1} \
  --total-frames 1048576 \
  --frames-per-batch 8192 --minibatch-size 512 --ppo-epochs 6 \
  --lr 1e-4 --ent-coef 0.005 --clip-eps 0.15 \
  --ckpt-every 2 \
  --save-dir {DRIVE_ROOT}/ckpts/stage2 \
  --tb-logdir {DRIVE_ROOT}/runs/stage2

## 7. Stage 3 — 일반화 완성: 전 알파벳, 긴 시퀀스

**Pool**: A~Z 전체 (`ALL_LETTERS`)  
**길이**: 4~6개 타겟  
**`--load-ckpt`**: Stage 2의 좋은 ckpt

LR 5e-5, entropy 0.003, clip 0.1 — 거의 finetune 수준의 보수적 셋업으로 기존 정책 미세 조정.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/sequence"
BEST_ITER_2 = 120  # ← Stage 2 학습 로그에서 가장 좋은 iter로 수정

import os
ckpt2 = f"{DRIVE_ROOT}/ckpts/stage2/ckpt_{BEST_ITER_2}.pt"
assert os.path.exists(ckpt2), f"Stage 2 ckpt 없음: {ckpt2}"
print("Stage 2 ckpt 로드:", ckpt2)

!python comm_train_battery.py \
  --grid-size 25 --n-agents 14 --max-steps 600 \
  --random-shape-pool ALL_LETTERS --random-path-length "4-6" \
  --shapes GROUND,A \
  --completion-reward 50.0 \
  --assigned-target-reward 0.1 --coverage-delta-reward 0.3 \
  --hover-penalty 0.05 --shaping-coef 0.5 \
  --initial-battery 1.0 --hover-battery-cost 0.002 --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.15 \
  --load-ckpt {ckpt2} \
  --total-frames 1572864 \
  --frames-per-batch 8192 --minibatch-size 512 --ppo-epochs 6 \
  --lr 5e-5 --ent-coef 0.003 --clip-eps 0.1 \
  --ckpt-every 2 \
  --save-dir {DRIVE_ROOT}/ckpts/stage3 \
  --tb-logdir {DRIVE_ROOT}/runs/stage3

## 8. TensorBoard (3개 stage 한 번에 보기)

각 stage curve가 누적으로 보임. Stage 2/3는 Stage 1보다 위 reward에서 시작해야 정상.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/drone_results/sequence/runs

## 9. 평가: 임의의 specific 시퀀스로 테스트

학습된 일반화 모델이 **사용자가 지정한 임의 시퀀스**를 잘 만드는지 확인. `comm_eval_battery.py`는 `--shapes`로 fixed 시퀀스를 받음. random-pool은 학습 전용이고 평가는 deterministic.

`TEST_SEQUENCE`를 자유롭게 바꿔서 여러 시퀀스 테스트 가능.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/sequence"
BEST_ITER_3 = 180  # ← Stage 3 학습 로그에서 best iter
TEST_SEQUENCE = "GROUND,B,A,Y,C"   # ← 원하는 임의 시퀀스로 자유 변경
TEST_TAG = "BAYC"                   # ← 파일명용 짧은 태그

!python comm_eval_battery.py \
  --ckpt {DRIVE_ROOT}/ckpts/stage3/ckpt_{BEST_ITER_3}.pt \
  --grid-size 25 --n-agents 14 --max-steps 600 \
  --shapes {TEST_SEQUENCE} \
  --completion-reward 50.0 \
  --initial-battery 1.0 --hover-battery-cost 0.002 --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.15 \
  --greedy --n-episodes 50 \
  --save-gif {DRIVE_ROOT}/gifs/demo_seq_{TEST_TAG}.gif \
  --out {DRIVE_ROOT}/evals/eval_seq_{TEST_TAG}.txt

## 10. (선택) 여러 시퀀스 한번에 평가

다양한 입력 시퀀스에 대해 일괄 평가해서 일반화 정도를 정량 확인. 결과는 모두 Drive에 누적 저장됨.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/sequence"
BEST_ITER_3 = 180  # ← Stage 3 best iter

TEST_BATTERY = [
    ("BAYC",    "GROUND,B,A,Y,C"),
    ("HELLO",   "GROUND,H,E,L,L,O"),
    ("MIT",     "GROUND,M,I,T"),
    ("unseen",  "GROUND,K,W,Q,Z"),   # Stage 1/2에서 거의 안 본 글자 위주
    ("longest", "GROUND,A,B,C,D,E,F,G,H"),
]

for tag, seq in TEST_BATTERY:
    print(f"\n===== Evaluating: {seq} =====")
    !python comm_eval_battery.py \
      --ckpt {DRIVE_ROOT}/ckpts/stage3/ckpt_{BEST_ITER_3}.pt \
      --grid-size 25 --n-agents 14 --max-steps 800 \
      --shapes {seq} \
      --completion-reward 50.0 \
      --initial-battery 1.0 --hover-battery-cost 0.002 --move-battery-cost 0.005 \
      --low-battery-move-penalty 0.15 \
      --greedy --n-episodes 30 \
      --save-gif {DRIVE_ROOT}/gifs/demo_seq_{tag}.gif \
      --out {DRIVE_ROOT}/evals/eval_seq_{tag}.txt

## 10b. (핵심) Random-sequence 일반화 정량 평가

위 셀 10은 **사용자가 고른 5개 시퀀스**만 본다. 진짜 일반화 정도를 보려면 **eval도 random pool로 매 에피소드마다 새 시퀀스를 샘플**해야 한다. 200~500 에피소드 돌리면 평균이 통계적으로 의미있어짐.

출력에 추가로 나오는 항목:
- **Per-length success**: 길이별(예: 2글자, 3글자, 5글자) 성공률 — 길어질수록 어려워지는지 확인
- **Per-letter success**: 각 글자가 포함된 에피소드의 평균 성공률 — 잘 못하는 글자 식별
- **Sample failed sequences**: 어떤 시퀀스에서 실패했는지 샘플 5개

⚠️ random eval에서는 `--save-gif`로 한 에피소드 GIF만 저장되니, 다양한 시퀀스 GIF가 필요하면 셀 10을 쓰면 됨. 이 셀은 통계 측정용.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/sequence"
BEST_ITER_3 = 180  # ← Stage 3 best iter

# Random-eval 구성 - 학습 때 본 pool과 같은 분포로 평가하면 in-distribution 일반화 측정.
# pool/length를 일부러 학습보다 더 다양/길게 잡으면 out-of-distribution 측정.
!python comm_eval_battery.py \
  --ckpt {DRIVE_ROOT}/ckpts/stage3/ckpt_{BEST_ITER_3}.pt \
  --grid-size 25 --n-agents 14 --max-steps 800 \
  --shapes GROUND,A \
  --random-shape-pool ALL_LETTERS --random-path-length "3-6" \
  --completion-reward 50.0 \
  --initial-battery 1.0 --hover-battery-cost 0.002 --move-battery-cost 0.005 \
  --low-battery-move-penalty 0.15 \
  --greedy --n-episodes 300 \
  --save-gif {DRIVE_ROOT}/gifs/demo_seq_random_sample.gif \
  --out {DRIVE_ROOT}/evals/eval_seq_random.txt

print("\n=== eval txt 결과 보기 ===")
!cat {DRIVE_ROOT}/evals/eval_seq_random.txt

## 11. Drive 산출물 점검

새 세션 복구 시에도 이 셀로 "지금 Drive에 뭐가 남아있나"만 확인하면 그대로 이어갈 수 있음.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/drone_results/sequence"

print("=== ckpts (stage별 개수) ===")
!for d in {DRIVE_ROOT}/ckpts/*/; do echo "$d -> $(ls $d 2>/dev/null | wc -l) ckpts"; done

print("\n=== 각 stage의 마지막 5개 ckpt ===")
!for d in {DRIVE_ROOT}/ckpts/*/; do echo "  $d:"; ls $d 2>/dev/null | sort -t_ -k2 -n | tail -5; done

print("\n=== gifs ===")
!ls -la {DRIVE_ROOT}/gifs/ 2>/dev/null

print("\n=== evals ===")
!ls -la {DRIVE_ROOT}/evals/ 2>/dev/null

---

## 런타임 끊겼을 때 복구 절차

1. 새 GPU 런타임 할당
2. **셀 1 (clone) → 셀 4 (Drive mount) → 셀 6 (pip)** 만 다시 실행
3. 셀 14 (verify-drive)로 어느 stage까지 ckpt 남아있는지 확인
4. 끊긴 지점부터 이어 학습: 해당 stage 셀의 `BEST_ITER_*`를 마지막 ckpt 번호로 맞추고 그 셀부터 실행

**중요**: ckpt가 `ckpt-every 2`로 매우 자주 저장되므로 최대 손실은 2 iter 분량 (~1-2분).

---

## Stage 사이 BEST_ITER 결정 팁

각 stage 학습 로그에서 `success` 와 `mean_ep_reward` 둘 다 안정적으로 높은 iter를 고를 것. 단 stage 1은 짧은 시퀀스라 success가 높게 나오지만, stage 2/3는 길어지면 success가 낮아질 수 있음. 그 경우 `mean_ep_reward`가 더 신뢰할 만한 지표.